<a href="https://colab.research.google.com/github/embark-cybertraining/embark-scratch-notebooks/blob/main/EDI_OBIS_usecase_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EDI_OBIS Use Case Goal

The goal of this workflow is to demonstrate interoperable access, inspection, and visualization of environmental and biodiversity data from multiple marine data repositories within the California Current Large Marine Ecosystem.

Additional code at end to investigate dataset bounding boxes and the combined boxes, and querying obis by larger polygon instead of area_id for california current

## More details about workflow and datasets

The workflow combines:

- particulate organic matter sampling data from the California Current Ecosystem Long-Term Ecological Research (CCE-LTER) program hosted by EDI
- marine species occurrence records from OBIS

Usecase info here: https://docs.google.com/document/d/1bKVoiHk6WYLtEDKojf4Rn7LIS-R9YAom/edit#heading=h.jx14u33ubg8w


## Datasets Used in the Workflow

### EDI Dataset

**Repository:** Environmental Data Initiative (EDI)  
**Dataset landing page:**  
https://portal.edirepository.org/nis/mapbrowse?packageid=knb-lter-cce.104.13

**Direct data access endpoint used in notebook:**  
https://pasta.lternet.edu/package/data/eml/knb-lter-cce/104/13/01cc01e9b988aea447b1dd417eca7938

### EDI Dataset Summary

This dataset contains particulate organic matter (POM) measurements collected by the California Current Ecosystem Long-Term Ecological Research (CCE-LTER) program.

The notebook uses:
- sampling coordinates
- sampling depth
- date/time
- cruise and station metadata
- particulate carbon and nitrogen measurements
- C/N molar ratios

These data provide environmental and biogeochemical context for comparison with marine biodiversity occurrence records.

---

### OBIS Dataset / API Query

**Repository:** Ocean Biodiversity Information System (OBIS)  
**API endpoint used in notebook:**  
https://api.obis.org/v3/occurrence

**OBIS area reference:**  
https://api.obis.org/v3/area

**OBIS source information:**  
https://obis.org/sources/

### OBIS Query Summary

The notebook queries OBIS occurrence records for:

- scientific name: *Pyrosoma atlanticum*
- marine region area ID: `40003`
- region: California Current Large Marine Ecosystem

The OBIS records provide:
- species occurrence locations
- occurrence dates
- source dataset identifiers
- AphiaIDs
- biodiversity observation context

These records are visualized independently and together with the EDI environmental sampling data to support spatial comparison across repositories.

# Setup code

In [23]:
# Setup/imports

import requests, folium, html
import geopandas as gpd #for bounding box things
from shapely.geometry import box
import pandas as pd
from IPython.display import display #used for better df display in notebook
from IPython.display import HTML #used for clickable links in species list table

SCI_NAME = 'Pyrosoma' #'Apolemia'
#"Pyrosoma",#"Pyrosoma atlanticum",'Apolemia'

AREA_ID = 40003 # corresponds to Marine Region for California Current

RECORD_LIMIT = 10000 # query up to this number at once,
# with checks for if acquired complete dataset

## EDI data

In [24]:
# Access EDI data
# Will use your own access key if supplied, otherwise uses public access

base_url = (
    "https://pasta.lternet.edu/package/data/eml/"
    "knb-lter-cce/104/13/01cc01e9b988aea447b1dd417eca7938"
)

url = base_url

try:
    from google.colab import userdata

    EDI_API_KEY = userdata.get("EDI_ACCESS_KEY")

    if EDI_API_KEY:
        url = f"{base_url}?key={EDI_API_KEY}"
        print("Using EDI access key")
    else:
        print("No EDI access key found, using public access")

except ImportError:
    print("Not running in Google Colab, using public access")

df_edi = pd.read_csv(url)

print(f"EDI records retrieved: {len(df_edi)}")

print("Show random 5 rows:")
df_edi.sample(5)

Using EDI access key
EDI records retrieved: 3295
Show random 5 rows:


,studyName,Datetime UTC,Transect,Station,Latitude (º),Longitude (º),R2R Event,CCE Event,Cast Number,Sample Type,Bottle Number,Depth (m),N (µg/L),C (µg/L),N (µmol/L),C (µmol/L),C/N Molar Ratio,Notes,Cruise,Associative Bottle Number
1348,P1208MV,2012-08-17 09:02:13,CYCLE4,NaN,34.315820,-123.458300,NaN,428.0,48,NaN,17.0,24.0,5.641682,34.911577,0.402783,2.906706,7.216550,NaN,CCE-P1208,NaN
924,P1106MV,2011-07-03 02:37:00,Upfront 2,NaN,33.787600,-121.777000,NaN,310.0,33,NaN,8.0,NaN,15.341110,92.283246,1.095266,7.683419,7.015115,NaN,CCE-P1106,NaN
950,P1106MV,2011-07-03 05:40:00,Upfront 2,NaN,33.825100,-121.725600,NaN,328.0,36,NaN,6.0,25.0,34.035120,166.690750,2.429910,13.878521,5.711537,NaN,CCE-P1106,NaN
2800,P1908,2019-08-30 09:05:51,Cycle4,NaN,34.899744,-124.774256,NaN,577.0,70,NaN,4.0,60.0,12.693145,33.868369,0.906217,2.819850,3.111672,20190830.0905.001,CCE-P1908,NaN
377,P0704TN,2007-04-18 09:18:18,Cycle 4,NaN,33.819000,-121.084000,NaN,460.0,43,NaN,14.0,30.0,16.463405,80.278930,1.175392,6.683951,5.686574,NaN,CCE-P0704,NaN


In [25]:
# Transform EDI data
#  to use Darwin Core names for lat and lon
# Remove rows with no lat/lon
# Remove rows with missing varibles of interest (particulate C or N values)


# variables of interest
vars_of_interest = [
    "N (µg/L)",
    "C (µg/L)"
]

print(f"EDI records retrieved (total): {len(df_edi)}")

# Rename lat/lon columns and clean missing coordinates
df_edi = df_edi.rename(columns={
    "Longitude (º)": "decimalLongitude",
    "Latitude (º)": "decimalLatitude"
})

df_edi[["decimalLatitude", "decimalLongitude"]] = (
    df_edi[["decimalLatitude", "decimalLongitude"]]
    .apply(pd.to_numeric, errors="coerce")
)

print("Rows removed - missing [lat,lon]:",
      df_edi[["decimalLatitude", "decimalLongitude"]]
      .isna().any(axis=1).sum())

df_edi = df_edi.dropna(
    subset=["decimalLatitude", "decimalLongitude"]
)
# ---- Vars of interest

print("Rows removed - missing "+str(vars_of_interest)+":",
      df_edi[vars_of_interest]
      .isna()
      .all(axis=1)
      .sum()
    )

# Drop rows where all target variables are missing
df_edi = df_edi.dropna(
        subset=vars_of_interest,
        how="all"
    )

print(f"Rows remaining: {len(df_edi)}")

EDI records retrieved (total): 3295
Rows removed - missing [lat,lon]: 17
Rows removed - missing ['N (µg/L)', 'C (µg/L)']: 0
Rows remaining: 3278


In [26]:
# Inspect EDI data

# do this todo clean where no target variable we re interesed in (drop if no C or N)

print("\nDatetime range:")
print(pd.to_datetime(df_edi["Datetime UTC"], errors="coerce").min())
print(pd.to_datetime(df_edi["Datetime UTC"], errors="coerce").max())

print("\nDepth range:")
print(pd.to_numeric(df_edi["Depth (m)"], errors="coerce").min())
print(pd.to_numeric(df_edi["Depth (m)"], errors="coerce").max())

## todo: add min/max of variables particulate C&N of interest


Datetime range:
2006-05-11 09:40:48
2024-03-17 10:08:00

Depth range:
1.0
801.0


## OBIS data

In [27]:
# Access OBIS data as DataFrame

response = requests.get(
    "https://api.obis.org/v3/occurrence",
    params={
        "scientificname": SCI_NAME,
        "areaid": AREA_ID,
        "size": RECORD_LIMIT,
    },
)

df_obis = pd.DataFrame(response.json()["results"])

print(f"OBIS records retrieved: {len(df_obis)}")

if len(df_obis) < RECORD_LIMIT:
    print("All matching records retrieved.")
else:
    print("Record limit reached.")



OBIS records retrieved: 610
All matching records retrieved.


In [28]:
# Clean, inspect, and summarize OBIS data

#Not all records show the datasetName or datasetID but that can be obtained
# from the obis identifier dataset_id using another query
def get_obis_dataset_title(dataset_id):
    try:
        r = requests.get(
            "https://api.obis.org/dataset",
            params={"datasetid": dataset_id},
            timeout=30,
        )
        r.raise_for_status()

        data = r.json()

        if isinstance(data, dict):
            records = data.get("results", data.get("data", []))
        else:
            records = data

        if records:
            return records[0].get("title", "")

    except Exception:
        pass

    return ""


# Add dataset titles from OBIS dataset API
dataset_titles = {
    dataset_id: get_obis_dataset_title(dataset_id)
    for dataset_id in df_obis["dataset_id"].dropna().unique()
}

df_obis["datasetTitle"] = df_obis["dataset_id"].map(dataset_titles)


df_obis["decimalLatitude"] = pd.to_numeric(
    df_obis["decimalLatitude"],
    errors="coerce"
)

df_obis["decimalLongitude"] = pd.to_numeric(
    df_obis["decimalLongitude"],
    errors="coerce"
)

# We only want records where there are lat/lons
df_obis = df_obis.dropna(
    subset=["decimalLatitude", "decimalLongitude"]
).copy()

preview_cols = [ #used for displaying df and in map popup when clicking point
    "scientificName",
    "scientificNameID",  # LSID
    "aphiaID",
    "decimalLatitude",
    "decimalLongitude",
    "eventDate",
    "minimumDepthInMeters",
    "maximumDepthInMeters",
    "occurrenceStatus",
    "datasetTitle",
    "occurrenceID",
]

preview_cols = [c for c in preview_cols if c in df_obis.columns]

# Display what we got
print("\nPreview of key fields (random 10 rows):")
display(df_obis[preview_cols].sample(10))




Preview of key fields (random 10 rows):


,scientificName,scientificNameID,aphiaID,decimalLatitude,decimalLongitude,eventDate,minimumDepthInMeters,maximumDepthInMeters,occurrenceStatus,datasetTitle,occurrenceID
524,Pyrosoma atlanticum,urn:lsid:marinespecies.org:taxname:137250,137250,33.493200,-119.592000,2015-05-15T08:21:20Z,25.00,30.00,present,Rockfish Recruitment and Ecosystem Assessment ...,440edcb5-e15e-4991-8088-3d999f8f041f
288,Pyrosoma atlanticum,urn:lsid:marinespecies.org:taxname:137250,137250,36.656300,-121.946300,2012-06-13T04:40:49Z,25.00,30.00,present,Rockfish Recruitment and Ecosystem Assessment ...,313aecd5-ae0b-4307-9ebc-613e68f9cd9a
311,Pyrosoma atlanticum,urn:lsid:marinespecies.org:taxname:137250,137250,32.709000,-117.333200,2012-05-25T11:07:08Z,25.00,30.00,present,Rockfish Recruitment and Ecosystem Assessment ...,9de29570-5582-495e-9e93-4a5c98a7c71b
97,Pyrosoma atlanticum,urn:lsid:marinespecies.org:taxname:137250,137250,36.988200,-122.458500,2014-06-03T07:03:39Z,25.00,30.00,present,Rockfish Recruitment and Ecosystem Assessment ...,781e089f-2252-451d-9909-c462e1ec5eed
587,Pyrosoma atlanticum,urn:lsid:marinespecies.org:taxname:137250,137250,36.741700,-121.976700,1992-05-12T07:39:00Z,25.00,30.00,present,Rockfish Recruitment and Ecosystem Assessment ...,18dfd514-229e-439d-8e58-dcf305731ac3
509,Pyrosoma atlanticum,urn:lsid:marinespecies.org:taxname:137250,137250,38.168200,-123.366700,2012-05-15T09:01:10Z,25.00,30.00,present,Rockfish Recruitment and Ecosystem Assessment ...,89aa026c-d206-4af2-888f-783ecc2b3410
331,Pyrosoma,urn:lsid:marinespecies.org:taxname:137224,137224,36.718418,-122.066223,2000-09-27T07:27:00Z,10.82,10.82,NaN,Video Annotation and Reference System (VARS) d...,NaN
255,Pyrosoma atlanticum,urn:lsid:marinespecies.org:taxname:137250,137250,36.976700,-122.588300,1995-05-30T04:03:00Z,25.00,30.00,present,Rockfish Recruitment and Ecosystem Assessment ...,142bda0e-0676-4836-b733-a0a098f71a74
263,Pyrosoma atlanticum,urn:lsid:marinespecies.org:taxname:137250,137250,32.718000,-118.461300,2015-05-13T08:43:01Z,25.00,30.00,present,Rockfish Recruitment and Ecosystem Assessment ...,2613bdc5-66db-4891-98af-498b3c86ee17
472,Pyrosoma atlanticum,urn:lsid:marinespecies.org:taxname:137250,137250,36.745300,-121.984300,2015-05-07T08:02:02Z,25.00,30.00,present,Rockfish Recruitment and Ecosystem Assessment ...,eb3fb1f1-6e7f-45b4-9f22-1365d7570b46


In [41]:
# Inspect data

from IPython.display import HTML, display

# Unique species with WoRMS page links
species_list_df = (
    df_obis[["scientificName", "scientificNameID", "aphiaID"]]
    .drop_duplicates()
    .sort_values(by="scientificName")
    .reset_index(drop=True)
)

species_list_df["WoRMS_link"] = species_list_df["aphiaID"].apply(
    lambda x: (
        f'<a href="https://marinespecies.org/aphia.php?p=taxdetails&id={int(x)}" '
        f'target="_blank">WoRMS {int(x)}</a>'
    )
    if pd.notna(x)
    else ""
)

# Count occurrenceStatus values with associated scientific names

occurrence_status_summary = (
    df_obis.groupby("occurrenceStatus")["scientificName"]
    .agg(
        count="size",
        unique_scientificNames=lambda x: sorted(x.dropna().unique())
    )
    .reset_index()
)

# Show unique occurrenceStatus values for each scientificName

species_occ_status = (
    df_obis[
        ["scientificName", "occurrenceStatus"]
    ]
    .fillna("Not Supplied")
    .value_counts()
    .reset_index(name="count")
    .reset_index(drop=True)
)

# Unique datasets with clickable OBIS dataset links
df_obis["datasetURL"] = (
    "https://obis.org/dataset/" + df_obis["dataset_id"].astype(str)
)

datasets = (
    df_obis[["dataset_id", "datasetTitle", "datasetURL"]]
    .drop_duplicates()
    .sort_values(by="datasetTitle")
    .reset_index(drop=True)
)

# Make datasetURL clickable
datasets["datasetURL"] = datasets["datasetURL"].apply(
    lambda x: f'<a href="{x}" target="_blank">{x}</a>'
)


# Display more info

print(f"\nTotal unique datasets: {len(datasets)}")
display(HTML(datasets.to_html(escape=False, index=False)))

print("Unique species in the dataset:")
display(HTML(species_list_df.to_html(escape=False, index=False)))

print("Unique occurrenceStatus values per scientificName:")
display(species_occ_status)

print("\nDataFrame cols")
print(df_obis.columns.tolist())



Total unique datasets: 7


dataset_id,datasetTitle,datasetURL
aa16d305-d413-4c4a-90be-b1ec3298d58d,CAS Invertebrate Zoology (IZ),https://obis.org/dataset/aa16d305-d413-4c4a-90be-b1ec3298d58d
92af50ad-c9d7-4fe7-b012-eae0e77233a2,Los Angeles Urban Ocean Expedition 2019,https://obis.org/dataset/92af50ad-c9d7-4fe7-b012-eae0e77233a2
71c2c816-7e94-40b9-8e28-8172d9c5fefb,NOAA Ocean Exploration seawater eDNA metabarcoding (EX2301),https://obis.org/dataset/71c2c816-7e94-40b9-8e28-8172d9c5fefb
89e23fc8-3f61-4480-9de3-358fe6eefe0b,National Museum of Natural History Invertebrate Zoology Collections,https://obis.org/dataset/89e23fc8-3f61-4480-9de3-358fe6eefe0b
5b6251f6-a7a5-4dc9-994d-c9504b54776f,"Rockfish Recruitment and Ecosystem Assessment Survey, Catch Data",https://obis.org/dataset/5b6251f6-a7a5-4dc9-994d-c9504b54776f
c5687a17-e454-40f9-9a4b-d04b2c812d74,UF Invertebrate Zoology,https://obis.org/dataset/c5687a17-e454-40f9-9a4b-d04b2c812d74
a419c8da-35ed-4b62-9709-39b56369c44e,Video Annotation and Reference System (VARS) database,https://obis.org/dataset/a419c8da-35ed-4b62-9709-39b56369c44e


Unique species in the dataset:


scientificName,scientificNameID,aphiaID,WoRMS_link
Pyrosoma,urn:lsid:marinespecies.org:taxname:137224,137224,WoRMS 137224
Pyrosoma,NaN,137224,WoRMS 137224
Pyrosoma atlanticum,urn:lsid:marinespecies.org:taxname:137250,137250,WoRMS 137250
Pyrosoma atlanticum,NaN,137250,WoRMS 137250


Unique occurrenceStatus values per scientificName:


,scientificName,occurrenceStatus,count
0,Pyrosoma atlanticum,present,477
1,Pyrosoma,Not Supplied,127
2,Pyrosoma atlanticum,Not Supplied,6



DataFrame cols
['basisOfRecord', 'class', 'classid', 'datasetID', 'date_end', 'date_mid', 'date_start', 'date_year', 'decimalLatitude', 'decimalLongitude', 'depth', 'eventDate', 'eventID', 'family', 'familyid', 'genus', 'genusid', 'geodeticDatum', 'higherClassification', 'identificationRemarks', 'institutionID', 'kingdom', 'kingdomid', 'license', 'locality', 'locationID', 'marine', 'materialSampleID', 'maximumDepthInMeters', 'minimumDepthInMeters', 'occurrenceID', 'occurrenceStatus', 'order', 'orderid', 'organismQuantity', 'organismQuantityType', 'parentEventID', 'phylum', 'phylumid', 'recordedBy', 'sampleSizeUnit', 'sampleSizeValue', 'scientificName', 'scientificNameID', 'species', 'speciesid', 'subfamily', 'subfamilyid', 'subphylum', 'subphylumid', 'taxonRank', 'type', 'verbatimIdentification', 'waterBody', 'id', 'dataset_id', 'node_id', 'dropped', 'absence', 'originalScientificName', 'aphiaID', 'flags', 'bathymetry', 'shoredistance', 'sst', 'sss', 'country', 'datasetName', 'eventRe

## Map EDI and OBIS together

In [ ]:
# Create reusable feature group for EDI points
edi_popup_cols = df_edi.columns.tolist()

edi_points = folium.FeatureGroup(name="EDI Points")

for _, row in df_edi.iterrows():
    popup_rows = [
        f"<tr><th>{col}</th><td>{row[col]}</td></tr>"
        for col in edi_popup_cols
        if pd.notna(row.get(col))
    ]

    popup = folium.Popup(
        f"<table>{''.join(popup_rows)}</table>",
        max_width=300
    )

    folium.CircleMarker(
        location=[row["decimalLatitude"], row["decimalLongitude"]],
        radius=5,
        color="blue",
        fill=True,
        fill_color="blue",
        fill_opacity=0.7,
        popup=popup,
    ).add_to(edi_points)

##------------------------------------------
#reusable OBIS points layer
# Use only preview columns in popup
obis_popup_cols = preview_cols

# Create reusable feature group for OBIS points
obis_points = folium.FeatureGroup(name="OBIS Points")

for _, row in df_obis.iterrows():
    popup_rows = [
        f"<tr><th>{col}</th><td>{row[col]}</td></tr>"
        for col in obis_popup_cols
        if col in df_obis.columns and pd.notna(row.get(col))
    ]

    popup = folium.Popup(
        f"<table>{''.join(popup_rows)}</table>",
        max_width=300
    )

    folium.CircleMarker(
        location=[row["decimalLatitude"], row["decimalLongitude"]],
        radius=5,
        color="red",
        fill=True,
        fill_color="red",
        fill_opacity=0.7,
        popup=popup,
    ).add_to(obis_points)


In [ ]:
# Map EDI and OBIS data together

m_combined = folium.Map(
    location=[37, -123],
    zoom_start=5,
    tiles="OpenStreetMap"
)

# add points already created
edi_points.add_to(m_combined)
obis_points.add_to(m_combined)


m_combined

# calculate bounds of each dataset and view BBs

In [30]:
from shapely.geometry import box
import geopandas as gpd

# Get EDI data bounding box polyong (WKT)

# Create a GeoDataFrame using OBIS longitude/latitude coordinates.
gdf_obis = gpd.GeoDataFrame(
    df_obis,
    geometry=gpd.points_from_xy(
        df_obis["decimalLongitude"],
        df_obis["decimalLatitude"]
    ),
    crs="EPSG:4326" # EPSG:4326 indicates WGS84 geographic coordinates in decimal degrees.
)

# Calculate the spatial extent of all points.
# total_bounds returns:
# [min_longitude, min_latitude, max_longitude, max_latitude]
minx, miny, maxx, maxy = gdf_obis.total_bounds

# Create a rectangular bounding box polygon from the extent.
# box(minx, miny, maxx, maxy) returns a Shapely polygon.
obis_bbox_polygon = box(minx, miny, maxx, maxy)

# Convert the polygon geometry to WKT (Well-Known Text).
# WKT coordinate order is:
# (longitude latitude)
obis_bbox_wkt = obis_bbox_polygon.wkt

print("OBIS bounding box WKT:")
print(obis_bbox_wkt)

OBIS bounding box WKT:
POLYGON ((-116.932998657 29.8670005798, -116.932998657 46.49201, -126.125685 46.49201, -126.125685 29.8670005798, -116.932998657 29.8670005798))


In [31]:
# Get EDI data bounding box polyong (WKT)

#Create a GeoDataFrame using EDI longitude/latitude coordinates.
gdf_edi = gpd.GeoDataFrame(
    df_edi,
    geometry=gpd.points_from_xy(
        df_edi["decimalLongitude"],
        df_edi["decimalLatitude"]
    ),
    crs="EPSG:4326"
)

# Calculate the spatial extent of all points.
# total_bounds returns:
# [min_longitude, min_latitude, max_longitude, max_latitude]
minx, miny, maxx, maxy = gdf_edi.total_bounds

# Create a rectangular bounding box polygon from the extent.
# box(minx, miny, maxx, maxy) returns a Shapely polygon.
edi_bbox_polygon = box(minx, miny, maxx, maxy)

# Convert the polygon geometry to WKT (Well-Known Text).
edi_bbox_wkt = edi_bbox_polygon.wkt

print("EDI bounding box WKT:")
print(edi_bbox_wkt)


EDI bounding box WKT:
POLYGON ((-120.018686 31.51848, -120.018686 36.559092, -130.547487 36.559092, -130.547487 31.51848, -120.018686 31.51848))


In [32]:
from shapely.ops import unary_union

# Create combined bounding box polygon
combined_bbox_polygon = unary_union([
    obis_bbox_polygon,
    edi_bbox_polygon
]).envelope

# Convert combined bounding box polygon to WKT
combined_bbox_wkt = combined_bbox_polygon.wkt
combined_bbox_wkt

'POLYGON ((-130.547487 29.8670005798, -116.932998657 29.8670005798, -116.932998657 46.49201, -130.547487 46.49201, -130.547487 29.8670005798))'

In [ ]:
# Create map centered on combined bounding box
m_combined = folium.Map(
    location=[
        (combined_bbox_polygon.bounds[1] + combined_bbox_polygon.bounds[3]) / 2,
        (combined_bbox_polygon.bounds[0] + combined_bbox_polygon.bounds[2]) / 2
    ],
    zoom_start=5,
    tiles="OpenStreetMap"
)

# Add points form before
edi_points.add_to(m_combined)
obis_points.add_to(m_combined)

# Add OBIS bounding box with WKT popup
folium.GeoJson(
    obis_bbox_polygon,
    name="OBIS Bounding Box",
    style_function=lambda x: {
        "color": "red",
        "weight": 2,
        "fill": False
    },
    popup=folium.Popup(obis_bbox_polygon.wkt, max_width=500)
).add_to(m_combined)

# Add EDI bounding box with WKT popup
folium.GeoJson(
    edi_bbox_polygon,
    name="EDI Bounding Box",
    style_function=lambda x: {
        "color": "blue",
        "weight": 2,
        "fill": False
    },
    popup=folium.Popup(edi_bbox_polygon.wkt, max_width=500)
).add_to(m_combined)

# Add combined bounding box with WKT popup
folium.GeoJson(
    combined_bbox_polygon,
    name="Combined Bounding Box",
    style_function=lambda x: {
        "color": "purple",
        "weight": 3,
        "fill": False
    },
    popup=folium.Popup(combined_bbox_polygon.wkt, max_width=500)
).add_to(m_combined)

# Add layer control
folium.LayerControl().add_to(m_combined)

# Add popup showing latitude/longitude anywhere clicked on the map
m_combined.add_child(folium.LatLngPopup())

m_combined

In [ ]:
# Save Folium map as HTML
m_combined.save(SCI_NAME+"-combined_map.html")

# testing additional functionality

# Looking at expanded BB for OBIS data

query using pyobis and the polyon of the previous obis query which corresponded to data from  area 40003 plus edi dataset bounds

In [33]:
!pip install pyobis

In [34]:
# Access OBIS data using pyobis and the combined bounding box
from pyobis import occurrences

#TEST_POLYGON = 'POLYGON ((-188.964844 4.565474, -101.25 4.565474, -101.25 55.178868, -188.964844 55.178868, -188.964844 4.565474))'
response = occurrences.search(
    scientificname=SCI_NAME,
    geometry=combined_bbox_wkt, #calculated from all the obis data returned from area 40003 plus edi dataset bounds
    size=RECORD_LIMIT
)

df_obis_expanded = response.execute()


print(f"OBIS records retrieved: {len(df_obis_expanded)}")

if len(df_obis_expanded) < RECORD_LIMIT:
    print("All matching records retrieved.")
else:
    print("Record limit reached.")



OBIS records retrieved: 621
All matching records retrieved.


In [35]:
# Find OBIS expanded records that were not present in the original dataset

# Use occurrenceID if available and unique
new_obis_records = df_obis_expanded[
    ~df_obis_expanded["occurrenceID"].isin(df_obis["occurrenceID"])
].copy()

print(f"New OBIS records found: {len(new_obis_records)}")

# Preview new records
new_obis_records.head()

New OBIS records found: 9


,basisOfRecord,class,classid,datasetID,date_end,date_mid,date_start,date_year,decimalLatitude,decimalLongitude,...,verbatimLongitude,habitat,occurrenceRemarks,otherCatalogNumbers,specificEpithet,georeferenceSources,verbatimCoordinates,footprintSRS,county,unaccepted
64,PreservedSpecimen,Thaliacea,22626,NaN,-1.285632e+11,-1.285632e+11,-1.285632e+11,1965.0,36.563850,-121.941900,...,121.9419° W,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
150,MaterialSample,Thaliacea,22626,NaN,1.880064e+11,1.880064e+11,1.880064e+11,1975.0,33.982500,-118.835000,...,NaN,NaN,{DigIn 2025-04-01: Expedition [10|velero] Vele...,24018. CH,NaN,https://digitallibrary.usc.edu/Share/acso7360o...,33°58'42'' 33°59'12'' N 118°49'30'' 118°50'42...,NaN,NaN,NaN
167,MachineObservation,Thaliacea,22626,https://marineinfo.org/id/dataset/6514,1.319414e+12,1.319414e+12,1.319414e+12,2011.0,33.004317,-121.527750,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,EPSG:4326,NaN,NaN
197,MachineObservation,Thaliacea,22626,https://marineinfo.org/id/dataset/6515,1.318896e+12,1.318896e+12,1.318896e+12,2011.0,35.395000,-127.741667,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,EPSG:4326,NaN,NaN
370,PreservedSpecimen,Thaliacea,22626,NaN,6.298560e+10,4.726080e+10,3.153600e+10,1971.0,33.428002,-118.427316,...,118.427316° W,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [36]:
# Stop notebook execution if no new OBIS records were found
if len(new_obis_records) == 0:
    raise SystemExit(
        f"No new OBIS records found for {SCI_NAME}. Stopping notebook execution."
    )

In [ ]:
# Rebuild combined map with new points included

m_combined_with_new = folium.Map(
    location=[
        new_obis_records["decimalLatitude"].mean(),
        new_obis_records["decimalLongitude"].mean()
    ],
    zoom_start=5,
    tiles="OpenStreetMap"
)

# Add existing reusable layers
edi_points.add_to(m_combined_with_new)
obis_points.add_to(m_combined_with_new)

# Create reusable layer for new OBIS points
new_obis_points = folium.FeatureGroup(
    name="New OBIS Points",
    show=True
)

# Add new OBIS points to the reusable layer
for _, row in new_obis_records.dropna(
    subset=["decimalLatitude", "decimalLongitude"]
).iterrows():

    popup_rows = [
        f"<tr><th>{col}</th><td>{row[col]}</td></tr>"
        for col in preview_cols
        if col in new_obis_records.columns and pd.notna(row.get(col))
    ]

    popup = folium.Popup(
        f"<table>{''.join(popup_rows)}</table>",
        max_width=300
    )

    folium.CircleMarker(
        location=[row["decimalLatitude"], row["decimalLongitude"]],
        radius=8,
        color="orange",
        fill=True,
        fill_color="orange",
        fill_opacity=1,
        popup=popup,
    ).add_to(new_obis_points)

# Add new-points layer to map
new_obis_points.add_to(m_combined_with_new)

# Zoom to new points
m_combined_with_new.fit_bounds([
    [
        new_obis_records["decimalLatitude"].min(),
        new_obis_records["decimalLongitude"].min()
    ],
    [
        new_obis_records["decimalLatitude"].max(),
        new_obis_records["decimalLongitude"].max()
    ]
])

#show the bounding box used for the full obis query (using pyobis)
folium.GeoJson(
    combined_bbox_polygon,
    name="Combined Bounding Box",
    style_function=lambda x: {
        "color": "purple",
        "weight": 3,
        "fill": False
    },
    popup=folium.Popup(
        combined_bbox_polygon.wkt,
        max_width=500
    )
).add_to(m_combined_with_new)

# Add layer control after ALL layers are added
folium.LayerControl(collapsed=False).add_to(
    m_combined_with_new
)

m_combined_with_new

In [38]:
# Save Folium map as HTML
m_combined_with_new.save(SCI_NAME+"-combined_map_with_new_points.html")